# Compare with Published Baselines

Compare our model against H&E, Virtual CODEX, and MICA baselines.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sys
sys.path.insert(0, '..')
from sparc.utils.metrics import c_index as our_c_index

sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
# Published baseline results
performance_data = {
    "BLCA": {"H&E": 0.59, "Virtual CODEX": 0.61, "MICA": 0.67},
    "BRCA": {"H&E": 0.70, "Virtual CODEX": 0.68, "MICA": 0.75},
    "COAD": {"H&E": 0.71, "Virtual CODEX": 0.68, "MICA": 0.76},
    "ESCA": {"H&E": 0.71, "Virtual CODEX": 0.71, "MICA": 0.79},
    "GBM":  {"H&E": 0.53, "Virtual CODEX": 0.57, "MICA": 0.61},
    "HNSC": {"H&E": 0.62, "Virtual CODEX": 0.63, "MICA": 0.68},
    "LGG":  {"H&E": 0.73, "Virtual CODEX": 0.71, "MICA": 0.78},
    "LIHC": {"H&E": 0.73, "Virtual CODEX": 0.70, "MICA": 0.79},
    "PAAD": {"H&E": 0.63, "Virtual CODEX": 0.65, "MICA": 0.69},
    "RCC":  {"H&E": 0.80, "Virtual CODEX": 0.76, "MICA": 0.85},
    "SKCM": {"H&E": 0.63, "Virtual CODEX": 0.60, "MICA": 0.69},
    "STAD": {"H&E": 0.65, "Virtual CODEX": 0.60, "MICA": 0.70},
}

# Cancer types that need to be combined for RCC
RCC_CANCERS = ['TCGA-KIRC', 'TCGA-KIRP', 'TCGA-KICH']

In [ ]:
# Load our run results
RUN_PATH = Path('../runs/image_only_20260204_084126')

fold_files = sorted(RUN_PATH.glob('fold_*_result.json'))
print(f"Found {len(fold_files)} folds")

# Cancer types we need to extract (matching the baseline data)
CANCER_MAPPING = {
    'BLCA': ['TCGA-BLCA'],
    'BRCA': ['TCGA-BRCA'],
    'COAD': ['TCGA-COAD'],
    'ESCA': ['TCGA-ESCA'],
    'GBM': ['TCGA-GBM'],
    'HNSC': ['TCGA-HNSC'],
    'LGG': ['TCGA-LGG'],
    'LIHC': ['TCGA-LIHC'],
    'PAAD': ['TCGA-PAAD'],
    'RCC': ['TCGA-KIRC', 'TCGA-KIRP', 'TCGA-KICH'],  # Combined
    'SKCM': ['TCGA-SKCM'],
    'STAD': ['TCGA-STAD'],
}

In [ ]:
# Compute per-cancer C-index for our run, averaged across folds
fold_cindices = {ct: [] for ct in CANCER_MAPPING.keys()}

for fold_file in fold_files:
    with open(fold_file) as f:
        data = json.load(f)
    
    patients = data.get('test_patient_results', [])
    if not patients:
        continue
    
    patients_df = pd.DataFrame(patients)
    
    for short_name, tcga_names in CANCER_MAPPING.items():
        # Get all patients from these cancer types
        subset = patients_df[patients_df['cancer_type'].isin(tcga_names)]
        
        if len(subset) >= 5 and subset['event'].sum() >= 1:
            c = our_c_index(
                subset['time'].values,
                subset['event'].values,
                subset['risk'].values
            )
            if not np.isnan(c):
                fold_cindices[short_name].append(c)

# Compute mean across folds
our_results = {}
for ct, cindices in fold_cindices.items():
    if cindices:
        our_results[ct] = {
            'mean': np.mean(cindices),
            'std': np.std(cindices),
            'n_folds': len(cindices)
        }
        print(f"{ct}: {our_results[ct]['mean']:.2f} +/- {our_results[ct]['std']:.2f} ({len(cindices)} folds)")

# Compute overall (macro-avg)
all_means = [v['mean'] for v in our_results.values()]
our_overall = np.mean(all_means)
print(f"\nOverall (macro-avg): {our_overall:.2f}")

In [ ]:
# Add our results to performance_data
for ct in performance_data.keys():
    if ct in our_results:
        performance_data[ct]['H&E (Ours)'] = our_results[ct]['mean']
    else:
        performance_data[ct]['H&E (Ours)'] = np.nan

# Add Overall
performance_data['MACRO-AVG'] = {
    'H&E': 0.67, 
    'Virtual CODEX': 0.66, 
    'MICA': 0.73,
    'H&E (Ours)': our_overall
}

# Convert to DataFrame
df = pd.DataFrame(performance_data).T
df

## MMP Comparison (6 cancers, same splits)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 11,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
})

# ── Data (original order: BRCA, BLCA, LUAD, STAD, CRC, KIRC) ──

# PANTHER — unimodal H&E (Mahmood lab)
panther_mean = np.array([0.6530, 0.5655, 0.6036, 0.6061, 0.6574, 0.7154])
panther_std  = np.array([0.0693, 0.0952, 0.0438, 0.0897, 0.1317, 0.2013])

# MMP per-cancer — multimodal RNA+H&E (Mahmood lab, trained per-cancer)
mmp_pc_mean = np.array([0.753, 0.628, 0.643, 0.580, 0.636, 0.748])
mmp_pc_std  = np.array([0.069, 0.064, 0.013, 0.071, 0.120, 0.099])

# MMP pan-cancer — multimodal RNA+H&E (Mahmood lab, retrained pan-cancer)
# CRC = 0.9 * COAD + 0.1 * READ
crc_mean_pan = 0.9 * 0.680 + 0.1 * 0.833
crc_std_pan  = np.sqrt((0.9 * 0.042)**2 + (0.1 * 0.146)**2)
mmp_pan_mean = np.array([0.713, 0.655, 0.636, 0.613, crc_mean_pan, 0.744])
mmp_pan_std  = np.array([0.103, 0.034, 0.098, 0.121, crc_std_pan,  0.135])

# Ours — pan-cancer, unimodal H&E
ours_mean = np.array([0.736, 0.651, 0.702, 0.636, 0.718, 0.838])
ours_std  = np.array([0.026, 0.049, 0.086, 0.038, 0.081, 0.055])

# ── Reorder to reverse alphabetical: STAD, LUAD, KIRC, CRC, BRCA, BLCA ──
cancers = ['STAD', 'LUAD', 'KIRC', 'CRC', 'BRCA', 'BLCA']
idx = [3, 2, 5, 4, 0, 1]

panther_mean = panther_mean[idx]
panther_std  = panther_std[idx]
mmp_pc_mean  = mmp_pc_mean[idx]
mmp_pc_std   = mmp_pc_std[idx]
mmp_pan_mean = mmp_pan_mean[idx]
mmp_pan_std  = mmp_pan_std[idx]
ours_mean    = ours_mean[idx]
ours_std     = ours_std[idx]

# Macro averages
panther_macro = panther_mean.mean()
mmp_pc_macro  = mmp_pc_mean.mean()
mmp_pan_macro = mmp_pan_mean.mean()
ours_macro    = ours_mean.mean()

# ── Plot ──
fig, ax = plt.subplots(figsize=(11, 5.5))

n = len(cancers)
x = np.arange(n)
w = 0.19

# Colors: grays for baselines, teal for ours
c_panther = '#B0B0B0'
c_mmp_pc  = '#7A7A7A'
c_mmp_pan = '#4A4A4A'
c_ours    = '#2B6B8A'

methods = [
    ('PANTHER (H&E)',            panther_mean, panther_std, c_panther),
    ('MMP per-cancer (RNA+H&E)', mmp_pc_mean,  mmp_pc_std,  c_mmp_pc),
    ('MMP pan-cancer (RNA+H&E)', mmp_pan_mean, mmp_pan_std, c_mmp_pan),
    ('SPARC-Risk (H&E)',               ours_mean,    ours_std,    c_ours),
]

offsets = [-1.5*w, -0.5*w, 0.5*w, 1.5*w]
all_bars = []

for (label, means, stds, color), offset in zip(methods, offsets):
    bars = ax.bar(x + offset, means, w, yerr=stds, capsize=2.5,
                  color=color, edgecolor='black', linewidth=0.4,
                  label=label, zorder=3,
                  error_kw={'linewidth': 0.8, 'capthick': 0.8})
    all_bars.append((bars, means))

# Value labels (rotated to fit 4 bars)
for bars, vals in all_bars:
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.008,
                f'{v:.2f}', ha='center', va='bottom', fontsize=6, fontweight='medium',
                rotation=90)

ax.set_xticks(x)
ax.set_xticklabels(cancers, fontsize=12, fontweight='medium')
ax.set_ylabel('C-index', fontsize=13, fontweight='medium')
ax.set_ylim(0.40, 1.02)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='y', labelsize=10)
ax.yaxis.set_major_locator(plt.MultipleLocator(0.05))
ax.grid(axis='y', alpha=0.25, linewidth=0.5, zorder=0)

ax.legend(fontsize=9, loc='upper center', bbox_to_anchor=(0.5, 1.12),
          ncol=4, frameon=False, handlelength=1.2, columnspacing=1.2)

plt.tight_layout()
plt.savefig('../figs/mmp_comparison.pdf', dpi=300, bbox_inches='tight')
plt.savefig('../figs/mmp_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# ── Print summary ──
delta_panther = ours_mean - panther_mean
delta_mmp_pc  = ours_mean - mmp_pc_mean
delta_mmp_pan = ours_mean - mmp_pan_mean

print(f'\n{"Cancer":<8} {"PANTHER":>12} {"MMP p-c":>12} {"MMP pan":>12} {"SPARC-Risk":>12} {"Δ vs MMP pan":>12}')
print('-' * 62)
for i, c in enumerate(cancers):
    print(f'{c:<8} {panther_mean[i]:>7.2f}±{panther_std[i]:.2f} '
          f'{mmp_pc_mean[i]:>7.2f}±{mmp_pc_std[i]:.2f} '
          f'{mmp_pan_mean[i]:>7.2f}±{mmp_pan_std[i]:.2f} '
          f'{ours_mean[i]:>7.2f}±{ours_std[i]:.2f} '
          f'{delta_mmp_pan[i]:>+8.2f}')
print('-' * 62)
print(f'{"Macro":<8} {panther_macro:>12.2f} {mmp_pc_macro:>12.2f} '
      f'{mmp_pan_macro:>12.2f} {ours_macro:>12.2f} {ours_macro-mmp_pan_macro:>+8.2f}')
print(f'\nOurs better than PANTHER in    {(delta_panther > 0).sum()}/{n} cancers')
print(f'SPARC-Risk better than MMP p-c in    {(delta_mmp_pc > 0).sum()}/{n} cancers')
print(f'SPARC-Risk better than MMP pan in    {(delta_mmp_pan > 0).sum()}/{n} cancers')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 11,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
})

# ── Data (original order: BRCA, BLCA, LUAD, STAD, CRC, KIRC) ──

# PANTHER — unimodal H&E (Mahmood lab)
panther_mean = np.array([0.6530, 0.5655, 0.6036, 0.6061, 0.6574, 0.7154])
panther_std  = np.array([0.0693, 0.0952, 0.0438, 0.0897, 0.1317, 0.2013])

# MMP per-cancer — multimodal RNA+H&E (Mahmood lab, trained per-cancer)
mmp_pc_mean = np.array([0.753, 0.628, 0.643, 0.580, 0.636, 0.748])
mmp_pc_std  = np.array([0.069, 0.064, 0.013, 0.071, 0.120, 0.099])

# Ours — pan-cancer, unimodal H&E
ours_mean = np.array([0.736, 0.651, 0.702, 0.636, 0.718, 0.838])
ours_std  = np.array([0.026, 0.049, 0.086, 0.038, 0.081, 0.055])

# ── Reorder to reverse alphabetical: STAD, LUAD, KIRC, CRC, BRCA, BLCA ──
cancers = ['STAD', 'LUAD', 'KIRC', 'CRC', 'BRCA', 'BLCA']
idx = [3, 2, 5, 4, 0, 1]

panther_mean = panther_mean[idx]
panther_std  = panther_std[idx]
mmp_pc_mean  = mmp_pc_mean[idx]
mmp_pc_std   = mmp_pc_std[idx]
ours_mean    = ours_mean[idx]
ours_std     = ours_std[idx]

# Macro averages
panther_macro = panther_mean.mean()
mmp_pc_macro  = mmp_pc_mean.mean()
ours_macro    = ours_mean.mean()

# ── Plot ──
fig, ax = plt.subplots(figsize=(10, 5.5))

n = len(cancers)
x = np.arange(n)
w = 0.25

# Colors: grays for baselines, teal for ours
c_panther = '#B0B0B0'
c_mmp_pc  = '#7A7A7A'
c_ours    = '#2B6B8A'

methods = [
    ('PANTHER (H&E)',            panther_mean, panther_std, c_panther),
    ('MMP (RNA+H&E)', mmp_pc_mean,  mmp_pc_std,  c_mmp_pc),
    ('SPARC-Risk (H&E)',               ours_mean,    ours_std,    c_ours),
]

offsets = [-w, 0, w]
all_bars = []

for (label, means, stds, color), offset in zip(methods, offsets):
    bars = ax.bar(x + offset, means, w, yerr=stds, capsize=3,
                  color=color, edgecolor='black', linewidth=0.4,
                  label=label, zorder=3,
                  error_kw={'linewidth': 0.8, 'capthick': 0.8})
    all_bars.append((bars, means))

# Value labels
for bars, vals in all_bars:
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.008,
                f'{v:.2f}', ha='center', va='bottom', fontsize=7, fontweight='medium')

ax.set_xticks(x)
ax.set_xticklabels(cancers, fontsize=12, fontweight='medium')
ax.set_ylabel('C-index', fontsize=13, fontweight='medium')
ax.set_ylim(0.40, 0.98)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='y', labelsize=10)
ax.yaxis.set_major_locator(plt.MultipleLocator(0.05))
ax.grid(axis='y', alpha=0.25, linewidth=0.5, zorder=0)

ax.legend(fontsize=10, loc='upper center', bbox_to_anchor=(0.5, 1.10),
          ncol=3, frameon=False, handlelength=1.2, columnspacing=1.5)

plt.tight_layout()
plt.savefig('../figs/mmp_comparison_no_pan.pdf', dpi=300, bbox_inches='tight')
plt.savefig('../figs/mmp_comparison_no_pan.png', dpi=300, bbox_inches='tight')
plt.show()

# ── Print summary ──
delta_panther = ours_mean - panther_mean
delta_mmp_pc  = ours_mean - mmp_pc_mean

print(f'\n{"Cancer":<8} {"PANTHER":>12} {"MMP p-c":>12} {"SPARC-Risk":>12} {"Δ vs MMP p-c":>12}')
print('-' * 50)
for i, c in enumerate(cancers):
    print(f'{c:<8} {panther_mean[i]:>7.2f}±{panther_std[i]:.2f} '
          f'{mmp_pc_mean[i]:>7.2f}±{mmp_pc_std[i]:.2f} '
          f'{ours_mean[i]:>7.2f}±{ours_std[i]:.2f} '
          f'{delta_mmp_pc[i]:>+8.2f}')
print('-' * 50)
print(f'{"Macro":<8} {panther_macro:>12.2f} {mmp_pc_macro:>12.2f} '
      f'{ours_macro:>12.2f} {ours_macro-mmp_pc_macro:>+8.2f}')
print(f'\nOurs better than PANTHER in    {(delta_panther > 0).sum()}/{n} cancers')
print(f'SPARC-Risk better than MMP p-c in    {(delta_mmp_pc > 0).sum()}/{n} cancers')